In [1]:
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / "experiments" / "features"
ADAPTED_DIR = FEATURE_DIR / "adapted"

DEVICE = "cpu"

print("Project root:", PROJECT_ROOT)
print("Feature directory exists:", FEATURE_DIR.exists())
print("Adapted directory exists:", ADAPTED_DIR.exists())

Project root: d:\DUK- Research Collab\Sanjay PhD\cnn-domain-adaptation-optimal-transport
Feature directory exists: True
Adapted directory exists: True


In [2]:
feature_data = torch.load(
    FEATURE_DIR / "office31_amazon_webcam_features.pt",
    map_location="cpu",
    weights_only=False
)

Xs_train = feature_data["Xs_train"].numpy()
Xs_test = feature_data["Xs_test"].numpy()

Xt_test = feature_data["Xt_test"].numpy()

ys_train = feature_data["ys_train"].numpy()
ys_test = feature_data["ys_test"].numpy()
yt_test = feature_data["yt_test"].numpy()

CLASSES = feature_data["classes"]

print("Source train:", Xs_train.shape)
print("Source test:", Xs_test.shape)
print("Target test:", Xt_test.shape)

print("Training labels:", ys_train.shape)
print("Source test labels:", ys_test.shape)
print("Target test labels:", yt_test.shape)

print("Number of classes:", len(CLASSES))

Source train: (2253, 512)
Source test: (564, 512)
Target test: (398, 512)
Training labels: (2253,)
Source test labels: (564,)
Target test labels: (398,)
Number of classes: 31


In [3]:
METHODS = {
    "EMD": "emd.pt",
    "Sinkhorn": "sinkhorn.pt",
    "Sinkhorn_LpL1": "sinkhorn_lpl1.pt",
    "Sinkhorn_L1L2": "sinkhorn_l1l2.pt"
}

adapted_features = {}

for method, filename in METHODS.items():

    file_path = ADAPTED_DIR / filename

    data = torch.load(
        file_path,
        map_location="cpu",
        weights_only=False
    )

    adapted_features[method] = data["Xs_train_adapted"].numpy()

    # Verify labels are consistent across methods
    assert np.array_equal(
        data["ys_train"].numpy(),
        ys_train
    ), f"Label mismatch in {method}"

    print(
        method,
        "| Shape:", adapted_features[method].shape,
        "| Finite:", np.isfinite(adapted_features[method]).all()
    )

EMD | Shape: (2253, 512) | Finite: True
Sinkhorn | Shape: (2253, 512) | Finite: True
Sinkhorn_LpL1 | Shape: (2253, 512) | Finite: True
Sinkhorn_L1L2 | Shape: (2253, 512) | Finite: True


In [4]:
classifier_config = {
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 2000,
    "random_state": 42
}

baseline_classifier = LogisticRegression(**classifier_config)

baseline_classifier.fit(
    Xs_train,
    ys_train
)

print("Baseline classifier trained!")

Baseline classifier trained!


In [5]:
def evaluate_classifier(model, X, y, dataset_name):

    predictions = model.predict(X)

    results = {
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y, predictions),
        "Macro F1": f1_score(
            y,
            predictions,
            average="macro"
        )
    }

    return results

In [6]:
baseline_results = []

baseline_results.append(
    evaluate_classifier(
        baseline_classifier,
        Xs_test,
        ys_test,
        "Amazon Source Test"
    )
)

baseline_results.append(
    evaluate_classifier(
        baseline_classifier,
        Xt_test,
        yt_test,
        "Webcam Target Test"
    )
)

baseline_df = pd.DataFrame(baseline_results)

baseline_df.round(4)

,Dataset,Accuracy,Balanced Accuracy,Macro F1
0,Amazon Source Test,0.8369,0.8335,0.8329
1,Webcam Target Test,0.6357,0.6604,0.6289


In [7]:
adapted_classifiers = {}
adapted_results = []

for method, X_adapted in adapted_features.items():

    print(f"\nTraining classifier: {method}")

    classifier = LogisticRegression(**classifier_config)

    classifier.fit(
        X_adapted,
        ys_train
    )

    adapted_classifiers[method] = classifier

    # Evaluate on held-out Webcam target test
    result = evaluate_classifier(
        classifier,
        Xt_test,
        yt_test,
        "Webcam Target Test"
    )

    result["Method"] = method
    adapted_results.append(result)

    print(
        f"Accuracy: {result['Accuracy']:.4f} | "
        f"Balanced Accuracy: {result['Balanced Accuracy']:.4f} | "
        f"Macro F1: {result['Macro F1']:.4f}"
    )


Training classifier: EMD
Accuracy: 0.5905 | Balanced Accuracy: 0.5952 | Macro F1: 0.5864

Training classifier: Sinkhorn
Accuracy: 0.5226 | Balanced Accuracy: 0.5202 | Macro F1: 0.5042

Training classifier: Sinkhorn_LpL1
Accuracy: 0.5251 | Balanced Accuracy: 0.5225 | Macro F1: 0.5077

Training classifier: Sinkhorn_L1L2
Accuracy: 0.5251 | Balanced Accuracy: 0.5227 | Macro F1: 0.5062


In [8]:
baseline_target = baseline_df[
    baseline_df["Dataset"] == "Webcam Target Test"
].iloc[0]

comparison_results = [
    {
        "Method": "Source Only",
        "Accuracy": baseline_target["Accuracy"],
        "Balanced Accuracy": baseline_target["Balanced Accuracy"],
        "Macro F1": baseline_target["Macro F1"]
    }
]

for result in adapted_results:
    comparison_results.append({
        "Method": result["Method"],
        "Accuracy": result["Accuracy"],
        "Balanced Accuracy": result["Balanced Accuracy"],
        "Macro F1": result["Macro F1"]
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df.round(4)

,Method,Accuracy,Balanced Accuracy,Macro F1
0,Source Only,0.6357,0.6604,0.6289
1,EMD,0.5905,0.5952,0.5864
2,Sinkhorn,0.5226,0.5202,0.5042
3,Sinkhorn_LpL1,0.5251,0.5225,0.5077
4,Sinkhorn_L1L2,0.5251,0.5227,0.5062


In [9]:
for metric in ["Accuracy", "Balanced Accuracy", "Macro F1"]:

    baseline_value = comparison_df.loc[
        comparison_df["Method"] == "Source Only",
        metric
    ].iloc[0]

    comparison_df[f"{metric} Improvement (pp)"] = (
        comparison_df[metric] - baseline_value
    ) * 100

comparison_df.round(4)

,Method,Accuracy,Balanced Accuracy,Macro F1,Accuracy Improvement (pp),Balanced Accuracy Improvement (pp),Macro F1 Improvement (pp)
0,Source Only,0.6357,0.6604,0.6289,0.0000,0.0000,0.0000
1,EMD,0.5905,0.5952,0.5864,-4.5226,-6.5190,-4.2458
2,Sinkhorn,0.5226,0.5202,0.5042,-11.3065,-14.0135,-12.4674
3,Sinkhorn_LpL1,0.5251,0.5225,0.5077,-11.0553,-13.7883,-12.1195
4,Sinkhorn_L1L2,0.5251,0.5227,0.5062,-11.0553,-13.7654,-12.2693
